## This Notebooks is for Testing of my agent codebases

In [1]:
import sys

PROJECT_ROOT = r'E:/WoodWorks_Ai'
sys.path.append(str(PROJECT_ROOT))

### 1. Agent_user

In [2]:
from schemas.agent_state import AgentState
from schemas.user.address import Address
from agents.agent_1_user.schema import UserRegistrationInput
from agents.agent_1_user import Agent1User
from integrations.database.engine import SessionLocal

state = AgentState()

db = SessionLocal()

agent = Agent1User(db)

user_input = UserRegistrationInput(
    name="Kuldeep Yadav",
    email="kuldeep@test.com",
    phone="9999999999",
    address=Address(
        line_1="123 Street",
        city="Delhi",
        state="Delhi",
        pincode="110001",
    ),
)

state = agent.run(state, user_input)

print(state)


DEBUG    Agent-1 started

INFO     Existing user found

INFO     User registration completed

workflow_state=<WorkflowState.USER_REGISTERED: 'USER_REGISTERED'> user_context=UserContext(id=1, name='Kuldeep Yadav', email='kuldeep@test.com', phone='9999999999', address=Address(line_1='123 Street', line_2=None, city='Delhi', state='Delhi', pincode='110001')) product_contexts=None human_requirements=None technical_spec=None pricing_context=None confirmation=ConfirmationFlags(human_requirements_confirmed=False, order_confirmed=False, human_confirmed_at=None, order_confirmed_at=None) exception=None


### 2. Agent_ProductSelection

In [3]:
from agents.agent_2_product.agent import Agent2Product
from agents.agent_2_product.schema import ProductSelectionInput
from schemas.base.enums import ProductCategory
from integrations.database.engine import SessionLocal

agent = Agent2Product(SessionLocal())

product_input = ProductSelectionInput(
    product_id=1,
    category=ProductCategory.STORAGE_UNIT,
    variant="Sliding Wardrobe",
)

state = agent.run(state, product_input)
print(state)

DEBUG    Agent-2 started

INFO     Product selected

workflow_state=<WorkflowState.PRODUCT_SELECTED: 'PRODUCT_SELECTED'> user_context=UserContext(id=1, name='Kuldeep Yadav', email='kuldeep@test.com', phone='9999999999', address=Address(line_1='123 Street', line_2=None, city='Delhi', state='Delhi', pincode='110001')) product_contexts=[ProductContext(product_id=1, category=<ProductCategory.STORAGE_UNIT: 'STORAGE_UNIT'>, variant='Sliding Wardrobe')] human_requirements=None technical_spec=None pricing_context=None confirmation=ConfirmationFlags(human_requirements_confirmed=False, order_confirmed=False, human_confirmed_at=None, order_confirmed_at=None) exception=None


### 3. Supervisor testing

In [4]:
from supervisor.supervisor import Supervisor
from schemas.agent_state import AgentState
from schemas.base.enums import WorkflowState

state = AgentState()

supervisor = Supervisor()

def print_decision(state):
    decision = supervisor.decide_next(state)
    print("Current State :", state.workflow_state)
    print("Next Action   :", decision["action"])
    print("-" * 40)
    return decision


print_decision(state)

state.workflow_state = WorkflowState.USER_REGISTERED
print_decision(state)

state.workflow_state = WorkflowState.PRODUCT_SELECTED
print_decision(state)

state.workflow_state = WorkflowState.HUMAN_REQUIREMENTS_CONFIRMED
print_decision(state)

state.workflow_state = WorkflowState.TECH_SPEC_FINALIZED
print_decision(state)

state.workflow_state = WorkflowState.ORDER_CONFIRMED
print_decision(state) 

DEBUG    Supervisor decision started

INFO     Supervisor decision made

Current State : WorkflowState.SESSION_STARTED
Next Action   : RUN_AGENT_1_USER
----------------------------------------


DEBUG    Supervisor decision started

INFO     Supervisor decision made

...

Current State : WorkflowState.TECH_SPEC_FINALIZED
Next Action   : RUN_AGENT_4_PRICING
----------------------------------------


DEBUG    Supervisor decision started

INFO     Workflow complete or paused

Current State : WorkflowState.ORDER_CONFIRMED
Next Action   : None
----------------------------------------


{'action': None,
 'state': AgentState(workflow_state=<WorkflowState.ORDER_CONFIRMED: 'ORDER_CONFIRMED'>, user_context=None, product_contexts=None, human_requirements=None, technical_spec=None, pricing_context=None, confirmation=ConfirmationFlags(human_requirements_confirmed=False, order_confirmed=False, human_confirmed_at=None, order_confirmed_at=None), exception=None)}

### 4. Agent_Human_Requirements

In [5]:
from agents.agent_3_human_requirements import Agent3HumanRequirements
from schemas.agent_state import AgentState
from schemas.base.enums import WorkflowState, ProductCategory
from schemas.product.product_context import ProductContext

state = AgentState(
    workflow_state=WorkflowState.PRODUCT_SELECTED,
    product_contexts=[
        ProductContext(
            product_id=1,
            category=ProductCategory.STORAGE_UNIT,
            variant="Wardrobe",
        )
    ],
)

agent = Agent3HumanRequirements()

state = agent.run(
    state,
    user_input={
        "usage": "home",
        "environment": "bedroom",
        "load_requirement": "heavy",
    },
)

print(state.workflow_state)
print(state.human_requirements)


DEBUG    Agent-3 (Human Requirements) started

INFO     Human requirements confirmed

WorkflowState.HUMAN_REQUIREMENTS_CONFIRMED
fields=[HumanRequirementField(key='usage', label='Intended Usage', value='home', value_type=<FieldValueType.STRING: 'STRING'>, unit=None, required=True), HumanRequirementField(key='load_requirement', label='Load Requirement', value='heavy', value_type=<FieldValueType.STRING: 'STRING'>, unit=None, required=True), HumanRequirementField(key='environment', label='Environment', value='bedroom', value_type=<FieldValueType.STRING: 'STRING'>, unit=None, required=True), HumanRequirementField(key='safety_priority', label='Safety Priority', value=False, value_type=<FieldValueType.BOOLEAN: 'BOOLEAN'>, unit=None, required=False), HumanRequirementField(key='aesthetic_focus', label='Aesthetic Focus', value=False, value_type=<FieldValueType.BOOLEAN: 'BOOLEAN'>, unit=None, required=False)]


### 5. AGent tech_spec

In [6]:
from agents.agent_4_tech_spec import Agent4TechSpec
from schemas.agent_state import AgentState
from schemas.base.enums import WorkflowState, FieldValueType
from schemas.human_requirements.requirements import HumanRequirements
from schemas.human_requirements.field import HumanRequirementField

state = AgentState(
    workflow_state=WorkflowState.HUMAN_REQUIREMENTS_CONFIRMED,
    human_requirements=HumanRequirements(
        fields=[
            HumanRequirementField(
                key="load_requirement",
                label="Load Requirement",
                value="HEAVY",
                value_type=FieldValueType.STRING,
                required=True,
            ),
            HumanRequirementField(
                key="environment",
                label="Environment",
                value="KITCHEN",
                value_type=FieldValueType.STRING,
                required=True,
            ),
            HumanRequirementField(
                key="safety_priority",
                label="Safety",
                value=True,
                value_type=FieldValueType.BOOLEAN,
                required=False,
            ),
        ]
    ),
)

agent = Agent4TechSpec()
state = agent.run(state)

print(state.workflow_state)
print(state.technical_spec)


DEBUG    Agent-4 (Tech Spec) started

INFO     Technical specification finalized

WorkflowState.TECH_SPEC_FINALIZED
fields=[TechnicalSpecField(key='material', value='PLYWOOD', unit=None, derived=True), TechnicalSpecField(key='finish', value='LAMINATE', unit=None, derived=True), TechnicalSpecField(key='thickness_mm', value=18, unit='mm', derived=True), TechnicalSpecField(key='waterproof', value=True, unit=None, derived=True), TechnicalSpecField(key='fire_resistant', value=True, unit=None, derived=True)] assumptions=[]
